# **SCAPES**

Follow this notebook to train your own SCAPES model and run it locally or online using the GUI. This notebook gives you a deep understanding of how SCAPES works, diving into its mechanisms in some detail. If you are only interested in training with default parameters and generating sounds, refer to the simplified notebook `tutorial_simplified.ipynb` instead.

This single notebook covers all three SCAPES tasks end-to-end:
1. **Dataprep** — extract audio atoms and precompute annotations
2. **Training** — train a FlowModel + LocalEncoder
3. **Inference** — reconstruct, semantically reconstruct, and interpolate audio

This notebook works in Colab out of the box. If running locally, we strongly recommend copying it into a clean directory, creating a virtual environment, and running it there. The notebook will install all dependencies, including the SCAPES code, automatically.

# **0. Preamble**

## Check if your runtime has a GPU

SCAPES uses a transformer architecture that benefits significantly from GPU acceleration. Check your GPU status using the following cell.

In [ ]:
!nvidia-smi

## Install the model

Run this cell to install SCAPES and its dependencies. After installation, you may be prompted to restart the kernel — if so, restart and run this cell again.

In [ ]:
# Clone the repo
!git clone https://github.com/cordutie/SCAPES.git
%cd SCAPES

# Install dependencies
!pip install -r requirements.txt

# Ensure Python can find the SCAPES package
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)

print("SCAPES setup complete.")

## Mount your Google Drive (optional)

If running in Colab, you may want to mount your Google Drive to transfer files. This is optional — you can also simply upload audio files by dragging and dropping them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# **1. Dataset Preparation**

In this section, we will extract EnCodec audio **atoms** (sequences of continuous embeddings) and precompute **semantic** embeddings (CLAP).


## Dataset Structure

First, make sure your data follows the structure below. Create a new directory for your dataset with two subdirectories: `raw/` (for your WAV files) and `config/` (for the configuration files). Copy the gin and CSV templates from `SCAPES/config/` into your `config/` directory to get started.

```
<dataset>/
├── raw/
│   ├── audio_file_1.wav
│   ├── ...
│   └── audio_file_N.wav
└── config/
    ├── dataprep.gin
    ├── training.gin
    └── cherry_picking.csv
```

## Configure your dataprep.gin

Once your dataset is set up, you can start editing the `.gin` configuration files. Below you will find a description of each parameter along with its possible values. Here are some key definitions to get you started:

- **EnCodec**: A model that compresses and decompresses audio. It can compress stereo audio at 48 kHz into a signal with 128 channels, each sampled at 150 Hz, and reconstruct the original audio from it.
- **EnCodec frame**: A single sample from EnCodec, consisting of 128 channels. One frame corresponds to 1/150 seconds of audio.
- **Atom**: SCAPES generates small chunks of encoded audio using EnCodec. These chunks are called *atoms*, in honor of Saint Arnau's pioneering work on texture sounds. A single atom spans several EnCodec frames.
- **Memory buffer**: When generating audio, SCAPES conditions on a list of past atoms. This sequence is called the memory buffer.
- **CLAP**: A model that captures semantic information from audio files.
- **Semantic embedding**: Semantic information extracted from each audio file, used to annotate what each atom contains. SCAPES uses this to understand context and decide what type of sounds to generate as a conditioning mechanism.

```Atom geometry
atoms.frames = 48          # number of frames in each atom
atoms.hop_frames = 15      # number of frames to be used as hop between atoms
atoms.crossfade_frames = 3 # number of frames used to overlap atoms

# Semantic conditioning
semantic.context_seconds = 1.0   # context window length used to capture semantics
semantic.random_extension = True # technique used to extend audios smaller than 7 seconds. True means stochastic repetition of subsegments of the original audio, while False means full repetition until 7 seconds are achieved.

# Structure features 
structure.features = [] # Technically one can use acoustic features to improve CLAP representation. This is currently deprecated. Leave as is.

# Precomputation (CLAP + structure)
precompute.batch_size = 128 # Batch size used when computing CLAP embeddings (decrease this number if you don't have much GPU memory)

# Split controls (optional)
splits.train_split = None # training split directory (if None, it will be chosen automatically)
splits.val_split   = None # validation split directory (if None, it will be chosen automatically)
splits.val_split_ratio = 0.1 # If the above are None, this controls the percentage of data used in each split

# Dataset sliding window
dataset.memory_buffer_atoms = 3 # number of atoms to be used in the memory buffer
dataset.hop_atoms = 1           # number of atoms between dataset points

# Misc
seed = 42       # use a seed if you want to reproduce your results
device = "auto" # device to be used by PyTorch
```

Once you have set your `dataprep.gin` file, continue with the following cells to start the data preparation process.

## Define the path for your dataset and initialize the dataprep functions

In [ ]:
# === CHANGE THIS ===
DATASET_PATH = "path/to/your/dataset" # <--------------------------- Change this to the path of your dataset

import sys, pathlib, torch
repo_root = pathlib.Path.cwd()
sys.path.insert(0, str(repo_root))

from SCAPES.data.config_loader import load_dataprep_config
from SCAPES.data.dataprep import atoms_maker, precompute_semantic_annotations, precompute_structure_annotations
from SCAPES.data.dataset import AtomSequenceDataset

dataset_path = pathlib.Path(DATASET_PATH).resolve()
config = load_dataprep_config(dataset_path / "config" / "dataprep.gin")
device = "cuda" if torch.cuda.is_available() else "cpu"
use_structure = bool(config.structure_features)

print(f"Dataset: {dataset_path}, Device: {device}")
print(f"Atoms frames: {config.atoms_frames}, hop: {config.atoms_hop_frames}")
print(f"Structure: {'enabled' if use_structure else 'disabled'}")

## Extract the atoms from your audio files

In [ ]:
# Step 1: Extract EnCodec atoms from raw audio
atoms_maker(str(dataset_path), config=config)
print("Atoms extracted.")

## Initialize your dataset and split it

In [ ]:
# Step 2: Initialize dataset and create train/validation split
dataset = AtomSequenceDataset(
    dataset_path=str(dataset_path),
    config=config,
    requested_keys=[
        "memory_buffer_latent", "target_latent",
        "memory_buffer_scale", "target_scale", "index"
    ],
    verbose=True
)
dataset.make_split(val_split=config.val_split_ratio, overwrite=True)
train_split, val_split = dataset.get_splits()
print(f"Train: {len(train_split)}, Val: {len(val_split)}")

## Precompute the semantic embeddings

In [ ]:
# Step 3: Precompute CLAP semantic annotations
precompute_semantic_annotations(
    dataset=dataset,
    batch_size=config.precompute_batch_size,
    device=device
)
print("Semantic annotations done.")

## Inspect a sample annotation (optional)

In [ ]:
# Check shapes of the precomputed annotations
anno_dir = dataset_path / "annotations"
idx = 0

semantic_path = anno_dir / "semantic" / f"semantic_{idx}.pt"
if semantic_path.exists():
    semantic = torch.load(semantic_path)
    print(f"Semantic shape: {semantic.shape}  (should be [n_atoms, 1024])")

if use_structure:
    struct_path = anno_dir / "structure" / f"structure_{idx}.pt"
    if struct_path.exists():
        struct = torch.load(struct_path)
        print(f"Structure shape: {struct.shape}  (should be [n_features, n_frames])")

## Visualize annotations (optional)

In [ ]:
from SCAPES.data.visualization import LatentSpaceExplorer

viz_dataset = AtomSequenceDataset(
    dataset_path=str(dataset_path),
    config=config,
    requested_keys=["target_semantic", "index"],
    device="cpu"
)
explorer = LatentSpaceExplorer(viz_dataset, max_samples_per_file=100)
explorer.plot_semantic(method="pca")
explorer.plot_semantic(method="tsne")

## Restart the kernel

After dataprep, restart the kernel to free memory before training. Then run the following cell to restore access to the SCAPES source code.

In [ ]:
%cd SCAPES
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)
print("SCAPES ready.")

# **2. Training**

Here we train the model and create the semantic cards that are used to run the model online. If not interested in running the model online, you can safely delete the cherry_picking.csv file in your dataset config folder.

## Load your dataset and define the path where you will save your model

In [ ]:
DATASET_PATH = "path/to/your/dataset" # <--------------------------- Change this to the path of your dataset
MODEL_PATH   = "/content/my_model" # <------------------------------ Change this to the path where you want to save the model

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW
from pathlib import Path

from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.data.config_loader import load_dataprep_config, load_training_config
from SCAPES.data.dataprep.semantic import build_semantics_folder
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.training.FlowModel_trainer import FlowTrainer
from SCAPES.models.factorization import LocalEncoder
from SCAPES.models.flow import FlowModel

# Optional: resume from 'latest' or 'best' checkpoint
resume_from = None  # or "latest", "best"

dataprep_config = load_dataprep_config(DATASET_PATH)
training_config = load_training_config(DATASET_PATH)
device = "cuda" if torch.cuda.is_available() else "cpu"

has_structure = (Path(DATASET_PATH) / "annotations" / "structure").exists()
requested_keys = [
    "memory_buffer_latent", "target_latent",
    "memory_buffer_scale", "target_scale",
    "target_semantic", "index"
]
if has_structure:
    requested_keys.insert(-1, "target_structure")

dataset = AtomSequenceDataset(
    dataset_path=DATASET_PATH,
    config=dataprep_config,
    requested_keys=requested_keys,
    device="cpu",
    verbose=True
)

train_split, val_split = dataset.get_splits()
train_loader = DataLoader(train_split, batch_size=training_config.batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_split, batch_size=training_config.batch_size, shuffle=False) if len(val_split) > 0 else None

print(f"Device: {device}, Model size: {training_config.size.upper()}")
print(f"Epochs: {training_config.epochs}, Batch: {training_config.batch_size}, LR: {training_config.learning_rate}")
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader) if val_loader else 0}")

## Build semantics folder from cherry-picking CSV (optional)

The `config/cherry_picking.csv` defines audio segments from which semantic cards can be extracted. These cards can then be used in the online GUI. If you are not interested in running the model online, you can safely delete this file.

Fill in the CSV following the template below, pointing to the filename and the time segment from which the semantics should be extracted:

| filename | start_sec | end_sec | flag | icon | description |
|---|---|---|---|---|---|
| secret_mars_recordings.wav | 0 | 5 | alien | 🛸 | alien sounds recorded on Mars |
|...|...|...|...|...|...|


In [ ]:
cherry_csv = Path(DATASET_PATH) / "config" / "cherry_picking.csv"
if cherry_csv.exists():
    build_semantics_folder(
        csv_path=cherry_csv,
        semantic_dir=Path(DATASET_PATH) / "annotations" / "semantic",
        output_dir=Path(MODEL_PATH) / "semantics",
        dataset=dataset,
    )

# Data-derived constants
if training_config.spectral_representation:
    frame_dim = 385
    frames_per_atom = dataset.atoms_frames // 2 + 1
else:
    frame_dim = 129
    frames_per_atom = dataset.atoms_frames
context_vector_dim = 1024

print(f"Frame dim: {frame_dim}, Frames/atom: {frames_per_atom}")
print(f"Memory buffer: {dataset.memory_buffer_atoms} atoms")
print(f"Structure dim: {dataset.structure_feature_dimension}")

# Initialize models
local_encoder = LocalEncoder(config=training_config, in_channels=frame_dim)
flow_model = FlowModel(
    config=training_config,
    frame_dim=frame_dim,
    context_vector_dim=context_vector_dim,
    num_past_atoms=dataset.memory_buffer_atoms,
    frames_per_atom=frames_per_atom,
    structure_dim=dataset.structure_feature_dimension,
    device=device
)

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)
print(f"LocalEncoder params: {count_params(local_encoder):,}")
print(f"FlowModel params:    {count_params(flow_model):,}")

optimizer = AdamW(
    list(flow_model.parameters()) + list(local_encoder.parameters()),
    lr=training_config.learning_rate
)
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

## Configure your training.gin and train

Change the training loop and model parameters in the `training.gin` file. Below is a reference with comments explaining each option:

```
# Architecture — either 'model.size' (picks from preset table) OR explicit params:
model.size = "large"
# model.d_model = 1024
# model.nhead = 16
# model.num_layers = 12
# model.dim_feedforward = 2048

# LocalEncoder (in_channels=129 is implicit — same as frame_dim, a codec constant)
local_encoder.hidden_dim = 256
local_encoder.time_entanglement = True
local_encoder.temporal_compression = 1

# Training hyperparameters
training.epochs = 75
training.batch_size = 128
training.learning_rate = 1e-4
training.past_dropout = 0.25
training.conditioning_dropout = 0.2
training.audio_val_freq = 15
training.val_nfe = 16
training.context_source = "clap"

# Checkpointing (0 = only best/last, no epoch checkpoints)
training.checkpoint_freq = 15
# training.save_resume_states = True   # saves optimizer states too (for resuming)

# Validation audio — filenames, "all", or "random=N":
# training.val_files = ["file1.wav", "file2.wav"]
# training.val_files = ["all"]
# training.val_files = ["random=3"]
# training.val_duration = 5.0

# Adversarial discriminator (True = 2-stage: standard FM + adversarial fine-tuning)
# training.use_discriminator = True
# training.stage2_epochs = 10

# Regularizers: list of [name, weight] pairs (e.g. [["time_phase", 0.1], ["fft_phase", 0.05]])
training.regularizers_and_weights = [["time_phase", 1], ["fft_phase", 0.1]]

# Spectral representation (experimental):
# When True, latents are transformed via rFFT → log-mag / cos / sin in the frequency domain,
# and the model learns the spectral velocity field directly. At inference, iFFT reconstructs
# the time-domain atoms. Incompatible with regularizers (phase is learned explicitly).
# training.spectral_representation = True

# Inference defaults
inference.cfg_scale = 1.0
```

In [ ]:
# Train
trainer = FlowTrainer(
    model=flow_model,
    local_encoder=local_encoder,
    train_loader=train_loader,
    val_loader=val_loader,
    dataset=dataset,
    processor=processor_48k,
    optimizer=optimizer,
    config=training_config,
    model_path=MODEL_PATH,
    resume_from=resume_from
)

trainer.train(
    epochs=training_config.epochs,
    audio_val_freq=training_config.audio_val_freq,
    val_nfe=training_config.val_nfe
)
print("Training complete!")

## Save your model (optional)

In [ ]:
zip_path = f"{MODEL_PATH.rstrip('/')}.zip"
!zip -r "{zip_path}" "{MODEL_PATH}"
print(f"Created: {zip_path}")

## Restart the kernel

After dataprep, restart the kernel to free memory before training. Then run the following cell to restore access to the SCAPES source code.

In [ ]:
%cd SCAPES
import sys, os
repo_path = os.getcwd()
if repo_path not in sys.path:
    sys.path.append(repo_path)
print("SCAPES ready.")

# **3. Inference**

There are two ways to run inference: using Python directly or using the online GUI. Details for both are provided below.

## **3.1. Online Inference**

In order to do online inference, you will need to upload your model to a Hugging Face model repository. This can be done directly in the browser using a free account.

First, clean up your model checkpoints and keep only the one you prefer. If in doubt, use the ones called `best`. You will also need to create a markdown file called `info.md` — write whatever you want in it, as this information will appear as the model description in the GUI.

```
<model>/
├── info.md
├── checkpoints/
│   ├── best_flow_model.pt
│   ├── best_local_encoder.pt
│   └── inference.gin
└── semantics/
    ├── mars_sounds.pt
    ├── ...
    └── semantics.csv
```

Once the structure is correct, create a Hugging Face account, look for the option to create a new model repository, and follow the instructions. This will generate a git repository. Then use the browser File Uploader to drag and drop your full model folder into it, fill in the commit message at the bottom of the page, and press Commit.

Once the upload is done, copy the link to your repository and paste it into the Add Models option in our online GUI available at: https://huggingface.co/spaces/cordutie/SCAPES-demo

Have fun!

## **3.2. Local Inference**

Load the trained model and generate audio in various modes.
All model and architecture settings come from `<model_dir>/checkpoints/inference.gin`
(generated during training).

**Key parameters:**
- `NFE`: Number of Function Evaluations for the ODE solver (higher = better quality, default: 32)
- `cfg_scale`: Classifier-free guidance scale (higher = more "typical" generations, default: 3.0)
- `TF` (Teacher Forcing):
  - `True` — use full ground-truth context (reconstruction)
  - `"partial"` — use 0.5 s of audio, then generate freely (semantic reconstruction)
  - `False` — fully generative (no teacher forcing)
- `decode_method`: `"ola_smooth"` (overlap-add with smoothing, default) or `"ola_linear"`

In [ ]:
import torch
from pathlib import Path
from IPython.display import Audio, display

from SCAPES.inference.FlowInference import (
    FlowInference,
    run_resynthesis_pipeline,
    run_batch_resynthesis_pipeline,
    run_interpolation_pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# === CHANGE THIS ===
model_path = "path/to/your/model"

# checkpoint: 'best' (best validation), 'last' (final), or an epoch number
engine = FlowInference(
    model_dir=model_path,
    device=device,
    verbose=True,
    checkpoint="best",
)
print(f"Loaded model from {model_path}")

### 3.2.1. Full reconstruction (teacher-forced)

Reconstructs audio from its full latent representation. The model gets ground-truth
atoms as context — measures autoencoder fidelity.

In [ ]:
audio_path = "path/to/your/audio.wav"
duration = 10

audio_tensor = engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000]
print("Original:", Path(audio_path).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine=engine,
    audio_path=audio_path,
    duration=duration,
    play=True,
    TF=True,
    NFE=16,  # fewer NFE is fine for TF reconstruction
)

### 3.2.2. Semantic reconstruction (partial TF)

Uses only the first 0.5 s to extract the CLAP embedding, then generates the rest
from scratch. Same semantic content, different acoustic realization.

In [ ]:
audio_path = "path/to/your/audio.wav"
duration = 10

audio_tensor = engine.load_audio_to_tensor(audio_path)[:, :, :duration * 48000]
print("Original:", Path(audio_path).stem)
display(Audio(audio_tensor.detach().cpu().numpy()[0], rate=48000))

output = run_resynthesis_pipeline(
    engine=engine,
    audio_path=audio_path,
    duration=duration,
    play=True,
    TF="partial",
    NFE=32,
)

### 3.2.3. Semantic interpolation

Interpolates between two audio files' semantic embeddings.
`stickyness` controls transition sharpness (1.0 = linear, higher = sharper midpoint).

In [ ]:
audio_1 = "path/to/start.wav"
audio_2 = "path/to/end.wav"

for path, label in [(audio_1, "Start"), (audio_2, "End")]:
    t = engine.load_audio_to_tensor(path)[:, :, :5 * 48000]
    print(f"{label}: {Path(path).stem}")
    display(Audio(t.detach().cpu().numpy()[0], rate=48000))

output = run_interpolation_pipeline(
    engine=engine,
    audio_path_1=audio_1,
    audio_path_2=audio_2,
    timeline_size=200,
    stay_time=20,
    stickyness=3.0,
    play=True,
    NFE=32,
    context_static=False,
)